In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
df = pd.read_csv(r"E:\sttu\student_performance_dataset.csv")
df



,Student_ID,Age,Gender,Attendance_Rate,Prior_GPA,LMS_Logins,Study_Hours_Per_Week,Assignment_Submission_Rate,Engagement_Score,Final_Academic_Performance
0,1,20,Female,91.27,2.45,194,26.6,96.70,43.33,High
1,2,18,Female,96.21,3.03,206,5.9,45.20,42.43,Medium
2,3,19,Male,96.83,1.16,119,23.1,84.67,68.21,High
3,4,24,Male,51.69,1.15,152,12.3,72.10,52.65,Medium
4,5,17,Male,60.51,3.10,99,1.6,41.70,48.59,Low
...,...,...,...,...,...,...,...,...,...,...
1995,1996,24,Male,53.20,2.08,119,32.8,31.83,96.37,Medium
1996,1997,18,Male,69.54,1.91,272,9.0,78.40,40.24,Low
1997,1998,20,Female,58.68,3.52,299,36.3,76.69,71.14,Low
1998,1999,22,Female,76.38,1.07,212,9.5,90.19,0.01,Low


In [3]:
TARGET_COL = "Final_Academic_Performance"   # original 3-class target

print(f"Using target column: {TARGET_COL}")
print("Unique values in target:", df[TARGET_COL].unique())


Using target column: Final_Academic_Performance
Unique values in target: ['High' 'Medium' 'Low']


In [4]:
def to_binary_label(val):
    if val == "Low":
        return "At_Risk"
    else:
        return "Not_At_Risk"

df["Performance_Binary"] = df[TARGET_COL].apply(to_binary_label)

print("\nAfter binarization:")
print(df["Performance_Binary"].value_counts())



After binarization:
Performance_Binary
Not_At_Risk    1628
At_Risk         372
Name: count, dtype: int64


In [5]:
numeric_features = [
    "Age",
    "Attendance_Rate",
    "Prior_GPA",
    "LMS_Logins",
    "Study_Hours_Per_Week",
    "Assignment_Submission_Rate",
    "Engagement_Score",
]

categorical_features = ["Gender"]

print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)

X = df[numeric_features + categorical_features]
y = df["Performance_Binary"]


Numeric features: ['Age', 'Attendance_Rate', 'Prior_GPA', 'LMS_Logins', 'Study_Hours_Per_Week', 'Assignment_Submission_Rate', 'Engagement_Score']
Categorical features: ['Gender']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # keep class balance in train & test
)

print("\nTrain size:", X_train.shape, "Test size:", X_test.shape)



Train size: (1600, 8) Test size: (400, 8)


In [7]:
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [8]:
svm_clf = SVC(probability=True)  # probability=True if you want predict_proba later

pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("clf", svm_clf),
    ]
)

param_grid = {
    "clf__C": [0.1, 1, 10, 100],
    "clf__kernel": ["rbf", "linear"],
    "clf__gamma": ["scale", "auto"],
    "clf__class_weight": ["balanced", None],
}

In [9]:
print("\n============================================================")
print("🔄 Training & tuning: SVM (Binary) – using f1_macro")
print("============================================================")

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="f1_macro",   # good for imbalanced binary
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("\n✅ Best Parameters for SVM (binary):", grid.best_params_)
print(f"✅ Best Cross-Validation f1_macro: {grid.best_score_:.4f}")



🔄 Training & tuning: SVM (Binary) – using f1_macro
Fitting 5 folds for each of 32 candidates, totalling 160 fits

✅ Best Parameters for SVM (binary): {'clf__C': 100, 'clf__class_weight': None, 'clf__gamma': 'scale', 'clf__kernel': 'rbf'}
✅ Best Cross-Validation f1_macro: 0.4941


In [10]:
y_pred = grid.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
print(f"\n🎯 Test Accuracy (SVM Binary): {test_acc:.4f}")

print("\n🎯 Test Results for SVM Binary")
print("-" * 30)
print("Classification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=grid.best_estimator_.classes_)
labels = grid.best_estimator_.classes_

print("\nConfusion Matrix (rows = true, cols = predicted):")
print("Labels order:", labels)
print(cm)



🎯 Test Accuracy (SVM Binary): 0.7225

🎯 Test Results for SVM Binary
------------------------------
Classification Report:
              precision    recall  f1-score   support

     At_Risk       0.18      0.14      0.15        74
 Not_At_Risk       0.81      0.86      0.83       326

    accuracy                           0.72       400
   macro avg       0.49      0.50      0.49       400
weighted avg       0.70      0.72      0.71       400


Confusion Matrix (rows = true, cols = predicted):
Labels order: ['At_Risk' 'Not_At_Risk']
[[ 10  64]
 [ 47 279]]


In [11]:
print("\nPretty Confusion Matrix:")
header_row = "True\\Pred   ".ljust(12)
for lab in labels:
    header_row += f"{lab:<12}"
print(header_row)

for i, true_lab in enumerate(labels):
    row = f"{true_lab:<12}"
    for j in range(len(labels)):
        row += f"{cm[i, j]:<12}"
    print(row)


Pretty Confusion Matrix:
True\Pred   At_Risk     Not_At_Risk 
At_Risk     10          64          
Not_At_Risk 47          279         


In [12]:
new_student = pd.DataFrame([{
    "Age": 20,
    "Attendance_Rate": 0.5,
    "Prior_GPA": 5,
    "LMS_Logins": 5,
    "Study_Hours_Per_Week": 1,
    "Assignment_Submission_Rate": 0,
    "Engagement_Score": 0,
    "Gender": "Female",
}])

pred_label = grid.predict(new_student)[0]
pred_proba = grid.predict_proba(new_student)[0]

print("\n🔍 Predicted binary performance for new student:", pred_label)
print("   Probabilities [class → prob]:")
for cls, prob in zip(grid.best_estimator_.classes_, pred_proba):
    print(f"   {cls}: {prob:.3f}")



🔍 Predicted binary performance for new student: At_Risk
   Probabilities [class → prob]:
   At_Risk: 0.172
   Not_At_Risk: 0.828
